In [31]:
# Data path for R library 

# /var/folders/lt/fs9cxhhn7_v8qvhkq56m6zh80000gn/T//RtmpQEZlmN/downloaded_packages

In [32]:
!playwright install chromium

In [33]:
import requests
from bs4 import BeautifulSoup
import json

# Use the standard readable URL
url = "https://www.fotmob.com/players/422685/bruno-fernandes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Next.js stores all the page data in this specific tag
    next_data_script = soup.find('script', id='__NEXT_DATA__')
    
    if next_data_script:
        # Load the text inside the script tag as a python dictionary
        json_data = json.loads(next_data_script.string)
        
        # The data you need is nested inside the props
        # Let's print the keys to help you navigate it
        print("Data extracted successfully. Top level keys:")
        print(json_data.keys())
        
        # To get you started, the core stats are usually under:
        # page_props = json_data['props']['pageProps']
    else:
        print("Could not find the __NEXT_DATA__ script tag.")
else:
    print(f"Request failed. Status code: {response.status_code}")

Data extracted successfully. Top level keys:
dict_keys(['props', 'page', 'query', 'buildId', 'isFallback', 'isExperimentalCompile', 'dynamicIds', 'gssp', 'appGip', 'scriptLoader'])


In [34]:
# Dig into the 'props' key
page_props = json_data['props']['pageProps']

print("Keys inside pageProps:")
print(page_props.keys())

# In many Next.js sites, the actual API response is cached inside a 'fallback' dictionary
if 'fallback' in page_props:
    fallback_data = page_props['fallback']
    
    # The keys in 'fallback' are usually the API URLs themselves
    # Let's get the first key and look inside
    api_url_key = list(fallback_data.keys())[0]
    player_data = fallback_data[api_url_key]
    
    print("\nSuccess! Here are the keys for the actual player data:")
    if isinstance(player_data, dict):
        print(player_data.keys())
else:
    # If no fallback, it might be directly in pageProps
    print("\nNo fallback found, check pageProps directly for stats/matches.")

Keys inside pageProps:
dict_keys(['data', 'fallback', 'translations'])

Success! Here are the keys for the actual player data:
dict_keys(['id', 'name', 'birthDate', 'contractEnd', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr'])


In [35]:
# Extract the list of recent matches
recent_matches = player_data.get('recentMatches', [])

if recent_matches:
    print(f"Found {len(recent_matches)} recent matches!")
    
    # Grab the very first match in the list to inspect its structure
    first_match = recent_matches[0]
    
    print("\nKeys inside a single match object:")
    print(first_match.keys())
    
    # Let's print out some of the juicy details if they exist
    print("\nSample Match Data:")
    print(f"Match ID: {first_match.get('id', 'Not found')}")
    print(f"Opponent: {first_match.get('opponent', {}).get('name', 'Not found')}")
    print(f"Player Rating: {first_match.get('rating', 'Not found')}")
    
else:
    print("No recent matches found in this dataset.")

Found 56 recent matches!

Keys inside a single match object:
dict_keys(['teamId', 'teamName', 'opponentTeamId', 'opponentTeamName', 'isHomeTeam', 'id', 'matchDate', 'matchPageUrl', 'leagueId', 'leagueName', 'stage', 'homeScore', 'awayScore', 'minutesPlayed', 'goals', 'assists', 'yellowCards', 'redCards', 'ratingProps', 'playerOfTheMatch', 'onBench'])

Sample Match Data:
Match ID: 4813697
Opponent: Not found
Player Rating: Not found


In [36]:
import requests
from bs4 import BeautifulSoup
import json

match_id = "4813697"

# Use the standard front-facing web URL instead of the dead API endpoint
url = f"https://www.fotmob.com/match/{match_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    next_data_script = soup.find('script', id='__NEXT_DATA__')
    
    if next_data_script:
        json_data = json.loads(next_data_script.string)
        page_props = json_data['props']['pageProps']
        
        print("Success, bypassed the 404. Keys inside the Next.js Match pageProps:")
        print(page_props.keys())
        
        # Digging into the fallback cache where the match stats usually live
        if 'fallback' in page_props:
            fallback_data = page_props['fallback']
            
            # Grab the first key in the fallback dictionary
            api_url_key = list(fallback_data.keys())[0]
            match_details = fallback_data[api_url_key]
            
            print("\nKeys inside the actual match details:")
            if isinstance(match_details, dict):
                print(match_details.keys())
                
                # Check if the specific lineup data is nested in here
                if 'content' in match_details and 'lineup' in match_details['content']:
                    print("\nFound the lineup data! Keys inside lineup:")
                    print(match_details['content']['lineup'].keys())
    else:
        print("Could not find the __NEXT_DATA__ script tag.")
else:
    print(f"Request failed. Status code: {response.status_code}")

Success, bypassed the 404. Keys inside the Next.js Match pageProps:
dict_keys(['general', 'header', 'nav', 'ongoing', 'hasPendingVAR', 'content', 'seo', 'fetchingLeagueData', 'ssr', 'fallback', 'translations'])

Keys inside the actual match details:
dict_keys(['matches'])


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/bs4/builder/__init__.py:314: RuntimeWarning: coroutine 'main' was never awaited
  for attr in list(attrs.keys()):


In [37]:
import pandas as pd

# Using your exact file path and fixing the Dtype warning
df = pd.read_csv('/Users/schoudhry/Desktop/mind-overnight/code/epl_match_details.csv', low_memory=False)

# Print all the column names so we can see the exact labels
print("Available columns in the dataset:")
columns = df.columns.tolist()

# Grouping them by 5 so it is easy to read
for i in range(0, len(columns), 5):
    print(columns[i:i+5])

Available columns in the dataset:
['match_id', 'match_round', 'league_id', 'league_name', 'league_round_name']
['parent_league_id', 'parent_league_season', 'match_time_utc', 'home_team_id', 'home_team']
['home_team_color', 'away_team_id', 'away_team', 'away_team_color', 'id']
['event_type', 'team_id', 'player_id', 'player_name', 'x']
['y', 'min', 'min_added', 'is_blocked', 'is_on_target']
['goal_crossed_y', 'expected_goals', 'expected_goals_on_target', 'shot_type', 'situation']
['period', 'is_own_goal', 'on_goal_shot_x', 'on_goal_shot_y', 'on_goal_shot_zoom_ratio']
['first_name', 'last_name', 'team_color', 'short_name', 'blocked_x']
['blocked_y', 'goal_crossed_z', 'full_name', 'is_saved_off_line', 'is_from_inside_box']


In [38]:
!playwright install chromium

In [39]:
!pip install sofascore-wrapper

In [40]:
import asyncio
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search

async def main():
    api = SofascoreAPI()
    
    # Search for the player
    search = Search(api, search_string="saka")
    raw_data = await search.search_all()
    
    # Loop through the massive results list
    for item in raw_data.get('results', []):
        
        # We only care if the result is actually a player
        if item.get('type') == 'player':
            entity = item['entity']
            player_id = entity['id']
            name = entity['name']
            team = entity['team']['name']
            
            print("Match Found:")
            print(f"Name: {name}")
            print(f"Player ID: {player_id}")
            print(f"Current Team: {team}")
            
            # Stop the loop after finding the first most relevant player
            break

    await api.close()

if __name__ == "__main__":
    asyncio.run(main())
    
    

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
import requests
import pandas as pd
import time

player_id = 934235 # Bukayo Saka

# 1. Get his recent matches
matches_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/0"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Origin": "https://www.sofascore.com",
    "Referer": "https://www.sofascore.com/"
}

response = requests.get(matches_url, headers=headers)
matches_data = response.json().get('events', [])

print(f"Found {len(matches_data)} recent matches. Extracting ratings...")

player_ratings = []

# 2. Loop through the matches to get his specific rating
# Limiting to 5 for the test run so we don't hit rate limits
for event in matches_data[:5]:
    match_id = event['id']
    home_team = event['homeTeam']['name']
    away_team = event['awayTeam']['name']
    matchup = f"{home_team} vs {away_team}"
    
    # Hit the lineups endpoint for this specific match
    lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
    lineup_resp = requests.get(lineup_url, headers=headers)
    
    rating = None
    if lineup_resp.status_code == 200:
        lineup_data = lineup_resp.json()
        
        # Search both home and away teams for our player
        for team_key in ['home', 'away']:
            if team_key in lineup_data:
                for player_node in lineup_data[team_key].get('players', []):
                    if player_node.get('player', {}).get('id') == player_id:
                        # Extract the rating
                        rating = player_node.get('statistics', {}).get('rating')
                        break
            if rating:
                break
                
    player_ratings.append({
        'Match_ID': match_id,
        'Matchup': matchup,
        'Rating': rating
    })
        
    # Be polite to the API to avoid 403 blocks
    time.sleep(0.5)

df_ratings = pd.DataFrame(player_ratings)
print("\nSaka's Match Ratings:")
print(df_ratings)

Found 0 recent matches. Extracting ratings...

Saka's Match Ratings:
Empty DataFrame
Columns: []
Index: []


In [ ]:

# %pip install cloudscraper

  Using cached cloudscraper-1.2.71-py2.py3-none-any.whl.metadata (19 kB)
Using cached cloudscraper-1.2.71-py2.py3-none-any.whl (99 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cloudscraper]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cloudscraper
import pandas as pd
import time

player_id = 934235 # Bukayo Saka

# cloudscraper automatically handles the headers and bot bypass
scraper = cloudscraper.create_scraper()

matches_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/0"

print(f"Fetching recent matches for player {player_id}...")
# Added a timeout of 10 seconds
response = scraper.get(matches_url, timeout=10)

# Check if the request was successful before trying to parse JSON
if response.status_code == 200:
    try:
        matches_data = response.json().get('events', [])
        print(f"Found {len(matches_data)} recent matches. Extracting ratings...")
    except ValueError:
        print("Failed to parse JSON. The site might be blocking the scraper.")
        matches_data = []
else:
    print(f"Failed to fetch matches. Status code: {response.status_code}")
    matches_data = []

player_ratings = []

# Limiting to 5 for the test run
for event in matches_data[:5]:
    match_id = event['id']
    home_team = event['homeTeam']['name']
    away_team = event['awayTeam']['name']
    matchup = f"{home_team} vs {away_team}"
    
    lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
    
    try:
        lineup_resp = scraper.get(lineup_url, timeout=10)
        rating = None
        
        if lineup_resp.status_code == 200:
            lineup_data = lineup_resp.json()
            
            for team_key in ['home', 'away']:
                if team_key in lineup_data:
                    for player_node in lineup_data[team_key].get('players', []):
                        if player_node.get('player', {}).get('id') == player_id:
                            rating = player_node.get('statistics', {}).get('rating')
                            break
                if rating:
                    break
        else:
            print(f"Warning: Failed to fetch lineup for {matchup} (Status: {lineup_resp.status_code})")
            
    except Exception as e:
        print(f"An error occurred while fetching {matchup}: {e}")
        rating = None
                
    player_ratings.append({
        'Match_ID': match_id,
        'Matchup': matchup,
        'Rating': rating
    })
        
    time.sleep(0.5)

df_ratings = pd.DataFrame(player_ratings)
print("\nSaka's Match Ratings:")
print(df_ratings)


Fetching recent matches for player 934235...
Failed to fetch matches. Status code: 403

Saka's Match Ratings:
Empty DataFrame
Columns: []
Index: []


In [42]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json

async def get_sofa_data(url):
    async with async_playwright() as p:
        # Launch browser
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        page = await context.new_page()
        
        # Go to the API URL directly
        await page.goto(url, wait_until="networkidle")
        
        # Pull the raw JSON text from the page body
        content = await page.inner_text("pre") # API responses usually render in a <pre> tag
        data = json.loads(content)
        
        await browser.close()
        return data

async def get_player_performance(player_id):
    # 1. Get the list of recent match events
    events_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/0"
    print(f"Fetching matches for {player_id}...")
    events_data = await get_sofa_data(events_url)
    events = events_data.get('events', [])
    
    match_list = []
    
    # 2. Get ratings for the last 5 matches
    for match in events[:5]:
        match_id = match['id']
        home = match['homeTeam']['name']
        away = match['awayTeam']['name']
        
        # Get the lineup/stats for this specific match
        lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
        print(f"Checking rating for: {home} vs {away}...")
        lineup_data = await get_sofa_data(lineup_url)
        
        rating = None
        # Loop through home and away lineups to find our guy
        for side in ['home', 'away']:
            if side in lineup_data:
                for p in lineup_data[side].get('players', []):
                    if p.get('player', {}).get('id') == player_id:
                        rating = p.get('statistics', {}).get('rating')
                        break
            if rating: break
            
        match_list.append({
            'Match': f"{home} vs {away}",
            'Rating': rating
        })
        
    return pd.DataFrame(match_list)

# Run it in your notebook
player_id = 934235 # Saka
df = await get_player_performance(player_id)
print("\n--- Saka's Recent Ratings ---")
print(df)

Fetching matches for 934235...
Checking rating for: Arsenal vs Tottenham Hotspur...
Checking rating for: Arsenal vs FC Bayern München...
Checking rating for: Chelsea vs Arsenal...
Checking rating for: Arsenal vs Brentford...
Checking rating for: Aston Villa vs Arsenal...

--- Saka's Recent Ratings ---
                          Match  Rating
0  Arsenal vs Tottenham Hotspur     6.9
1  Arsenal vs FC Bayern München     6.7
2            Chelsea vs Arsenal     7.4
3          Arsenal vs Brentford     8.2
4        Aston Villa vs Arsenal     6.5


In [45]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime

# Expanded Position Map to catch RM, LM, etc.
POSITION_MAP = {
    'RW': ['lb', 'left back', 'lwb', 'lbo'],
    'RM': ['lb', 'left back', 'lwb', 'lbo'],
    'LW': ['rb', 'right back', 'rwb', 'rbo'],
    'LM': ['rb', 'right back', 'rwb', 'rbo'],
    'ST': ['cb', 'centre back', 'center back', 'dc'],
    'CF': ['cb', 'centre back', 'center back', 'dc'],
    'CAM': ['dm', 'cm', 'defensive midfield', 'central midfield'],
}

async def get_sofa_json(page, url):
    await page.goto(url, wait_until="networkidle")
    # Using inner_text on 'body' is often safer if 'pre' isn't explicitly rendered
    content = await page.inner_text("body")
    return json.loads(content)

async def get_multi_season_matchups(player_id, stop_year=2025):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        page = await context.new_page()
        
        all_matchups = []
        last_match_id = 0 
        reached_end = False
        
        while not reached_end:
            url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/{last_match_id}"
            print(f"--- Fetching batch before ID: {last_match_id if last_match_id != 0 else 'Present'} ---")
            
            try:
                data = await get_sofa_json(page, url)
                events = data.get('events', [])
            except Exception as e:
                print(f"Failed to fetch batch: {e}")
                break
            
            if not events: 
                print("No more events found.")
                break
            
            for match in events:
                match_date = datetime.fromtimestamp(match.get('startTimestamp'))
                if match_date.year < stop_year:
                    reached_end = True
                    break
                
                match_id = match['id']
                lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
                
                try:
                    lineup_data = await get_sofa_json(page, lineup_url)
                    
                    target_player = None
                    target_side = None
                    
                    # 1. Correctly find which side the player is on
                    for side in ['home', 'away']:
                        players_list = lineup_data.get(side, {}).get('players', [])
                        for p_node in players_list:
                            if p_node.get('player', {}).get('id') == player_id:
                                target_player = p_node
                                target_side = side
                                break
                        if target_side: break
                    
                    # 2. Extract stats and find direct opponents
                    if target_player and target_player.get('statistics', {}).get('rating'):
                        rating = target_player['statistics']['rating']
                        # SofaScore roles can be 'RW', 'F', 'M', etc.
                        role = target_player.get('role', 'ST').upper()
                        print(f"Match: {match_id} | Detected Role: {role} | Rating: {rating}")
                        
                        opp_side = 'away' if target_side == 'home' else 'home'
                        look_for = POSITION_MAP.get(role, ['cb', 'lb', 'rb']) # Fallback to all defenders
                        
                        for opp in lineup_data.get(opp_side, {}).get('players', []):
                            opp_role = opp.get('role', '').lower()
                            # Search for any of our positional keywords in the opponent's role
                            if any(pos in opp_role for pos in look_for):
                                all_matchups.append({
                                    'Date': match_date.date(),
                                    'Opponent': opp.get('player', {}).get('name'),
                                    'Opponent_Role': opp_role,
                                    'Rating': rating,
                                    'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}"
                                })
                except Exception as e:
                    # Silently skip matches with missing lineup data
                    continue
            
            # Pagination: Move to the next batch
            last_match_id = events[-1]['id']
            print(f"Batch complete. Total matchups found so far: {len(all_matchups)}")
            if reached_end: break
            
        await browser.close()
        return pd.DataFrame(all_matchups)

# Executing for Saka
df = await get_multi_season_matchups(934235, stop_year=2025)

if not df.empty:
    print("\n--- Successful Extraction ---")
    print(df.head(10))
else:
    print("\nStill empty. Check if the player has ratings for these specific matches.")

--- Fetching batch before ID: Present ---
Match: 14025066 | Detected Role: ST | Rating: 6.9
Match: 14566717 | Detected Role: ST | Rating: 6.7
Match: 14025099 | Detected Role: ST | Rating: 7.4
Match: 14025126 | Detected Role: ST | Rating: 8.2
Match: 14025156 | Detected Role: ST | Rating: 6.5
Match: 14566746 | Detected Role: ST | Rating: 6.4
Match: 14025183 | Detected Role: ST | Rating: 8.5
Match: 14025220 | Detected Role: ST | Rating: 6.6
Match: 14970878 | Detected Role: ST | Rating: 7.3
Match: 14025234 | Detected Role: ST | Rating: 7.7
Match: 14025254 | Detected Role: ST | Rating: 6.6
Match: 14025274 | Detected Role: ST | Rating: 7.2
Match: 14025019 | Detected Role: ST | Rating: 6.6
Match: 15268210 | Detected Role: ST | Rating: 6.5
Match: 14025082 | Detected Role: ST | Rating: 7.2
Match: 14566612 | Detected Role: ST | Rating: 6.9
Match: 14025098 | Detected Role: ST | Rating: 7.7
Match: 14025185 | Detected Role: ST | Rating: 6.4
Match: 15359739 | Detected Role: ST | Rating: 7.4
Match: 1

In [46]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime

# We define matchups based on pitch zones
# Zone mapping: 1=GK, 2=Def, 3=Mid, 4=Att
# Side mapping: L=Left, R=Right, C=Center
def get_direct_opponents(player_pos_str, players_list):
    """
    Finds direct opponents based on pitch coordinates.
    Saka (RW) usually operates in the Right-Attacking zone.
    His direct opponent is the Left-Defending zone.
    """
    opponents = []
    
    # Saka is usually 'F' or 'M' on the Right
    # In SofaScore, tactical positions are often strings like 'right'
    p_pos = player_pos_str.lower()
    
    for p in players_list:
        opp_role = p.get('role', '').lower()
        opp_pos = p.get('position', '').lower() # D, M, F
        
        # LOGIC: If Attacker is on the RIGHT, Opponent is on the LEFT (from their perspective)
        if 'right' in p_pos:
            # Look for Left Backs / Left Wing Backs
            if 'left' in opp_role or 'left' in opp_pos:
                opponents.append(p.get('player', {}).get('name'))
        # LOGIC: If Attacker is CENTER (ST), Opponent is CENTER (CB)
        elif 'centre' in p_pos or 'center' in p_pos or 'st' in p_pos:
            if 'centre' in opp_role or 'center' in opp_role or 'cb' in opp_role:
                opponents.append(p.get('player', {}).get('name'))
        # LOGIC: If Attacker is on the LEFT, Opponent is on the RIGHT
        elif 'left' in p_pos:
            if 'right' in opp_role or 'right' in opp_pos:
                opponents.append(p.get('player', {}).get('name'))
                
    return opponents

async def get_sofa_json(page, url):
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)

async def get_multi_season_matchups(player_id, stop_year=2025):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0...")
        page = await context.new_page()
        
        all_matchups = []
        last_match_id = 0 
        
        url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/{last_match_id}"
        data = await get_sofa_json(page, url)
        events = data.get('events', [])
        
        for match in events:
            match_date = datetime.fromtimestamp(match.get('startTimestamp'))
            if match_date.year < stop_year: break
            
            match_id = match['id']
            lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
            
            try:
                lineup_data = await get_sofa_json(page, lineup_url)
                target_player = None
                target_side = None
                
                # 1. Find Saka and his EXACT tactical position string
                for side in ['home', 'away']:
                    for p_node in lineup_data.get(side, {}).get('players', []):
                        if p_node.get('player', {}).get('id') == player_id:
                            target_player = p_node
                            target_side = side
                            break
                
                if target_player and target_player.get('statistics', {}).get('rating'):
                    rating = target_player['statistics']['rating']
                    # Use 'position' or 'role' - whatever is available
                    tactical_pos = target_player.get('role') or target_player.get('position') or "ST"
                    
                    opp_side = 'away' if target_side == 'home' else 'home'
                    opponents = get_direct_opponents(tactical_pos, lineup_data.get(opp_side, {}).get('players', []))
                    
                    # If the coordinate logic fails, fall back to grabbing ALL defenders
                    if not opponents:
                        opponents = [o.get('player', {}).get('name') for o in lineup_data.get(opp_side, {}).get('players', []) if o.get('position') == 'D']

                    for opp_name in opponents:
                        all_matchups.append({
                            'Date': match_date.date(),
                            'Opponent': opp_name,
                            'Rating': rating,
                            'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}",
                            'Saka_Pos': tactical_pos
                        })
            except: continue
            
        await browser.close()
        return pd.DataFrame(all_matchups)

df = await get_multi_season_matchups(934235, stop_year=2025)
print(df.head(10))

         Date           Opponent  Rating                         Match  \
0  2025-11-23        Kevin Danso     6.9  Arsenal vs Tottenham Hotspur   
1  2025-11-23    Cristian Romero     6.9  Arsenal vs Tottenham Hotspur   
2  2025-11-23   Micky van de Ven     6.9  Arsenal vs Tottenham Hotspur   
3  2025-11-23        Pedro Porro     6.9  Arsenal vs Tottenham Hotspur   
4  2025-11-26     Josip Stanišić     6.7  Arsenal vs FC Bayern München   
5  2025-11-26    Dayot Upamecano     6.7  Arsenal vs FC Bayern München   
6  2025-11-26       Jonathan Tah     6.7  Arsenal vs FC Bayern München   
7  2025-11-26      Konrad Laimer     6.7  Arsenal vs FC Bayern München   
8  2025-11-26  Raphael Guerreiro     6.7  Arsenal vs FC Bayern München   
9  2025-11-26        Min-jae Kim     6.7  Arsenal vs FC Bayern München   

  Saka_Pos  
0        M  
1        M  
2        M  
3        M  
4        M  
5        M  
6        M  
7        M  
8        M  
9        M  


In [48]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime

async def get_sofa_json(page, url):
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)

async def get_multi_season_matchups(player_id, stop_year=2024):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0...")
        page = await context.new_page()
        
        all_matchups = []
        last_match_id = 0 
        reached_stop_date = False
        
        while not reached_stop_date:
            url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/{last_match_id}"
            print(f"\n--- Batch Search: Matches before ID {last_match_id if last_match_id != 0 else 'Present'} ---")
            
            try:
                data = await get_sofa_json(page, url)
                events = data.get('events', [])
            except: break
                
            if not events: break

            for match in events:
                match_date = datetime.fromtimestamp(match.get('startTimestamp'))
                if match_date.year < stop_year:
                    reached_stop_date = True
                    break
                
                match_id = match['id']
                lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
                
                try:
                    lineup_data = await get_sofa_json(page, lineup_url)
                    target_player = None
                    target_side = None
                    
                    # 1. Find our player
                    for side in ['home', 'away']:
                        for p_node in lineup_data.get(side, {}).get('players', []):
                            if p_node.get('player', {}).get('id') == player_id:
                                target_player = p_node
                                target_side = side
                                break
                    
                    if target_player and target_player.get('statistics', {}).get('rating'):
                        rating = target_player['statistics']['rating']
                        opp_side = 'away' if target_side == 'home' else 'home'
                        
                        # 2. GREEDY FILTER: Grab EVERY defender ('D') on the opposing team
                        defenders = []
                        for opp in lineup_data.get(opp_side, {}).get('players', []):
                            # Position 'D' is the universal tag for any defender in SofaScore
                            if opp.get('position') == 'D':
                                defenders.append(opp.get('player', {}).get('name'))
                        
                        if defenders:
                            print(f"Match found: {match['homeTeam']['name']} vs {match['awayTeam']['name']} | Defenders: {len(defenders)}")
                            for d_name in defenders:
                                all_matchups.append({
                                    'Date': match_date.date(),
                                    'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}",
                                    'Opponent': d_name,
                                    'Rating': rating
                                })
                except: continue
            
            # Pagination
            last_match_id = events[-1]['id']
            print(f"Total Rows in Memory: {len(all_matchups)}")
            if reached_stop_date: break
                
        await browser.close()
        return pd.DataFrame(all_matchups)

# Run it!
df_final = await get_multi_season_matchups(934235, stop_year=2024)

if not df_final.empty:
    df_final.to_csv('saka_matchups_raw.csv', index=False)
    print("\nSUCCESS: CSV saved with data!")
    print(df_final.head())
else:
    print("\nStill empty. This usually means the player didn't have ratings in the events found.")


--- Batch Search: Matches before ID Present ---
Match found: Arsenal vs Tottenham Hotspur | Defenders: 4
Match found: Arsenal vs FC Bayern München | Defenders: 8
Match found: Chelsea vs Arsenal | Defenders: 7
Match found: Arsenal vs Brentford | Defenders: 7
Match found: Aston Villa vs Arsenal | Defenders: 7
Match found: Club Brugge KV vs Arsenal | Defenders: 8
Match found: Arsenal vs Wolverhampton | Defenders: 6
Match found: Everton vs Arsenal | Defenders: 8
Match found: Arsenal vs Crystal Palace | Defenders: 5
Match found: Arsenal vs Brighton & Hove Albion | Defenders: 5
Match found: Arsenal vs Aston Villa | Defenders: 7
Match found: Bournemouth vs Arsenal | Defenders: 7
Match found: Arsenal vs Liverpool | Defenders: 7
Match found: Chelsea vs Arsenal | Defenders: 7
Match found: Nottingham Forest vs Arsenal | Defenders: 6
Match found: Inter vs Arsenal | Defenders: 8
Match found: Arsenal vs Manchester United | Defenders: 8
Match found: Brentford vs Arsenal | Defenders: 7
Match found: A

In [49]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime

async def get_sofa_json(page, url):
    """Fetches and parses JSON directly from the body to bypass UI rendering."""
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)

async def scrape_player_matchups(player_id, stop_year=2024):
    async with async_playwright() as p:
        # Launching the stealth browser
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        page = await context.new_page()
        
        all_matchups = []
        last_match_id = 0 
        reached_stop_date = False
        
        while not reached_stop_date:
            url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/{last_match_id}"
            print(f"\n--- Traversing: Fetching batch before ID {last_match_id if last_match_id != 0 else 'Present'} ---")
            
            try:
                data = await get_sofa_json(page, url)
                events = data.get('events', [])
            except Exception as e:
                print(f"Error fetching batch: {e}")
                break
                
            if not events:
                print("No more events found in history.")
                break

            for match in events:
                match_date = datetime.fromtimestamp(match.get('startTimestamp'))
                
                # Condition to stop once we hit your target year
                if match_date.year < stop_year:
                    reached_stop_date = True
                    break
                
                match_id = match['id']
                lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
                
                try:
                    lineup_data = await get_sofa_json(page, lineup_url)
                    target_player_rating = None
                    target_side = None
                    
                    # 1. Identify which side our attacker is on and grab their rating
                    for side in ['home', 'away']:
                        for p_node in lineup_data.get(side, {}).get('players', []):
                            if p_node.get('player', {}).get('id') == player_id:
                                # Only proceed if the attacker actually played (has a rating)
                                target_player_rating = p_node.get('statistics', {}).get('rating')
                                target_side = side
                                break
                        if target_side: break
                    
                    # 2. If the attacker played, find the ACTIVE defenders on the other team
                    if target_player_rating:
                        opp_side = 'away' if target_side == 'home' else 'home'
                        active_opponents = []
                        
                        for opp in lineup_data.get(opp_side, {}).get('players', []):
                            # FILTER: Must be a Defender ('D') AND must have a match rating (meaning they played)
                            opp_rating = opp.get('statistics', {}).get('rating')
                            if opp.get('position') == 'D' and opp_rating is not None:
                                active_opponents.append(opp.get('player', {}).get('name'))
                        
                        if active_opponents:
                            print(f"Match Logged: {match['homeTeam']['name']} vs {match['awayTeam']['name']} | Active Defenders: {len(active_opponents)}")
                            for d_name in active_opponents:
                                all_matchups.append({
                                    'Date': match_date.date(),
                                    'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}",
                                    'Opponent': d_name,
                                    'Attacker_Rating': target_player_rating
                                })
                except Exception:
                    continue # Skip matches with missing data (e.g. friendlies or data errors)
            
            # Pagination: Move the anchor to the oldest match in the current batch
            last_match_id = events[-1]['id']
            print(f"Total Rows Processed: {len(all_matchups)}")
            
            if reached_stop_date:
                break
                
        await browser.close()
        return pd.DataFrame(all_matchups)

# --- EXECUTION ---
# Using Saka's ID: 934235 (Change stop_year to 2021 if you want the full 5-season history)
df_results = await scrape_player_matchups(934235, stop_year=2024)

if not df_results.empty:
    filename = "Saka_vs_Active_Defenders.csv"
    df_results.to_csv(filename, index=False)
    print(f"\nSUCCESS: Data saved to {filename}")
    print(df_results.head(15))
else:
    print("\nExtraction failed: The resulting dataset is empty.")


--- Traversing: Fetching batch before ID Present ---
Match Logged: Arsenal vs Tottenham Hotspur | Active Defenders: 4
Match Logged: Arsenal vs FC Bayern München | Active Defenders: 6
Match Logged: Chelsea vs Arsenal | Active Defenders: 4
Match Logged: Arsenal vs Brentford | Active Defenders: 6
Match Logged: Aston Villa vs Arsenal | Active Defenders: 5
Match Logged: Club Brugge KV vs Arsenal | Active Defenders: 6
Match Logged: Arsenal vs Wolverhampton | Active Defenders: 4
Match Logged: Everton vs Arsenal | Active Defenders: 4
Match Logged: Arsenal vs Crystal Palace | Active Defenders: 4
Match Logged: Arsenal vs Brighton & Hove Albion | Active Defenders: 3
Match Logged: Arsenal vs Aston Villa | Active Defenders: 5
Match Logged: Bournemouth vs Arsenal | Active Defenders: 4
Match Logged: Arsenal vs Liverpool | Active Defenders: 4
Match Logged: Chelsea vs Arsenal | Active Defenders: 7
Match Logged: Nottingham Forest vs Arsenal | Active Defenders: 4
Match Logged: Inter vs Arsenal | Active 

In [50]:
import pandas as pd

# 1. Load your clean data
df = pd.read_csv('/Users/schoudhry/Desktop/mind-overnight/code/futbolV1/Saka_vs_Active_Defenders.csv')

# 2. Basic Aggregation
# We calculate the Mean rating and the Count of games for every defender
defender_stats = df.groupby('Opponent')['Attacker_Rating'].agg(['mean', 'count']).reset_index()
defender_stats.columns = ['Defender', 'Avg_Attacker_Rating', 'Match_Count']

# 3. Apply Bayesian Weighting
C = df['Attacker_Rating'].mean() # Overall average
m = 2 # Minimum matches for high confidence

defender_stats['Weighted_Difficulty'] = (
    (defender_stats['Match_Count'] * defender_stats['Avg_Attacker_Rating']) + (m * C)
) / (defender_stats['Match_Count'] + m)

# 4. Sort: Lower ratings mean the defender was MORE successful at stopping him
defender_rankings = defender_stats.sort_values(by='Weighted_Difficulty', ascending=True)

print(f"--- Attacker Difficulty Rankings (Lower = Harder Defender) ---")
print(defender_rankings.head(10))

--- Attacker Difficulty Rankings (Lower = Harder Defender) ---
              Defender  Avg_Attacker_Rating  Match_Count  Weighted_Difficulty
19          Ezri Konsa                 6.55            2             6.804918
94     Victor Lindelof                 6.55            2             6.804918
24         Hugo Siquet                 6.40            1             6.839891
61       Matheus Nunes                 6.40            1             6.839891
38        Joel Ordóñez                 6.40            1             6.839891
48       Kyriani Sabbe                 6.40            1             6.839891
71          Nathan Ake                 6.40            1             6.839891
75       Nico O'Reilly                 6.40            1             6.839891
82       Radu Drăguşin                 6.40            1             6.839891
1   Abdukodir Khusanov                 6.40            1             6.839891


In [51]:
import asyncio
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search

async def get_player_id(name_query):
    api = SofascoreAPI()
    search = Search(api, search_string=name_query)
    results = await search.search_all()
    await api.close()
    
    # Filter for the first 'player' entity in the results
    for item in results.get('results', []):
        if item.get('type') == 'player':
            return item['entity']['id'], item['entity']['name']
    return None, None

# --- USER INPUT ---
player_name = input("Enter the player you want to analyze (e.g. Harry Kane): ")
p_id, full_name = await get_player_id(player_name)

if p_id:
    print(f"Found {full_name} (ID: {p_id}). Starting the scrape...")
    
    # Now we call your previous function using this dynamic ID
    # Note: Make sure your scrape_player_matchups function is defined in a cell above
    df_new = await scrape_player_matchups(p_id, stop_year=2024)
    
    if not df_new.empty:
        csv_name = f"{full_name.replace(' ', '_')}_vs_Defenders.csv"
        df_new.to_csv(csv_name, index=False)
        print(f"Extraction complete! Saved to {csv_name}")
else:
    print("Could not find that player. Try being more specific.")

Found Harry Kane (ID: 108579). Starting the scrape...

--- Traversing: Fetching batch before ID Present ---
Match Logged: FC Bayern München vs SC Freiburg | Active Defenders: 5
Match Logged: Arsenal vs FC Bayern München | Active Defenders: 6
Match Logged: FC Bayern München vs FC St. Pauli | Active Defenders: 6
Match Logged: 1. FC Union Berlin vs FC Bayern München | Active Defenders: 4
Match Logged: VfB Stuttgart vs FC Bayern München | Active Defenders: 5
Match Logged: FC Bayern München vs Sporting CP | Active Defenders: 7
Match Logged: FC Bayern München vs 1. FSV Mainz 05 | Active Defenders: 5
Match Logged: 1. FC Heidenheim vs FC Bayern München | Active Defenders: 5
Match Logged: Red Bull Salzburg vs FC Bayern München | Active Defenders: 7
Match Logged: FC Bayern München vs VfL Wolfsburg | Active Defenders: 5
Match Logged: 1. FC Köln vs FC Bayern München | Active Defenders: 3
Match Logged: RB Leipzig vs FC Bayern München | Active Defenders: 4
Match Logged: FC Bayern München vs Royale U

In [53]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search
import nest_asyncio

nest_asyncio.apply()

async def get_sofa_json(page, url):
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)

async def run_analysis():
    # 1. Dynamic Player Search
    target_query = input("Enter the player to analyze (e.g., Saka or Harry Kane): ")
    
    api_wrapper = SofascoreAPI()
    search_engine = Search(api_wrapper, search_string=target_query)
    search_results = await search_engine.search_all()
    
    player_id, full_name = None, None
    for item in search_results.get('results', []):
        if item.get('type') == 'player':
            player_id = item['entity']['id']
            full_name = item['entity']['name']
            break
    
    await api_wrapper.close()
    
    if not player_id:
        print("Could not find that player; please try again.")
        return

    print(f"\n--- Researching {full_name} (ID: {player_id}) ---\n")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent="Mozilla/5.0")
        
        # Pull the recent event list
        events_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/0"
        events_data = await get_sofa_json(page, events_url)
        events = events_data.get('events', [])
        
        match_history = []
        
        for match in events[:10]: # Looking at the last 10 matches for the report
            match_id = match['id']
            lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
            
            try:
                lineup_data = await get_sofa_json(page, lineup_url)
                
                target_data = None
                target_side = None
                
                # Identify our attacker
                for side in ['home', 'away']:
                    for p_node in lineup_data.get(side, {}).get('players', []):
                        if p_node.get('player', {}).get('id') == player_id:
                            target_data = p_node
                            target_side = side
                            break
                    if target_side: break
                
                if target_data and target_data.get('statistics', {}).get('rating'):
                    attacker_rating = target_data['statistics']['rating']
                    opp_side = 'away' if target_side == 'home' else 'home'
                    opp_team_name = match[f'{opp_side}Team']['name']
                    
                    # Gather all active defenders for the Back Line Average
                    active_defenders = []
                    for opp in lineup_data.get(opp_side, {}).get('players', []):
                        d_rating = opp.get('statistics', {}).get('rating')
                        if opp.get('position') == 'D' and d_rating:
                            active_defenders.append({
                                'name': opp['player']['name'],
                                'rating': d_rating
                            })
                    
                    if active_defenders:
                        backline_avg = sum(d['rating'] for d in active_defenders) / len(active_defenders)
                        
                        # Print the humanized Match Log entry
                        print(f"Match: {match['homeTeam']['name']} vs {match['awayTeam']['name']}")
                        print(f"  > {full_name} Rating: {attacker_rating}")
                        print(f"  > Rating vs Back Line: {backline_avg:.2f}")
                        
                        for defender in active_defenders:
                            match_history.append({
                                'Defender': defender['name'],
                                'Team': opp_team_name,
                                'Defender_Rating': defender['rating'],
                                'Attacker_Rating': attacker_rating,
                                'Backline_Avg': backline_avg
                            })
                            print(f"    - vs {defender['name']} ({opp_team_name}): {attacker_rating} | {defender['rating']}")
                        print("-" * 30)

            except Exception: continue
            
        await browser.close()

    # 2. Aggregation and Bayesian Math
    if match_history:
        df = pd.DataFrame(match_history)
        global_avg = df['Attacker_Rating'].mean()
        m = 2 # Confidence factor
        
        stats = df.groupby(['Defender', 'Team']).agg({
            'Attacker_Rating': 'mean',
            'Defender_Rating': 'mean',
            'Defender': 'count'
        }).rename(columns={'Defender': 'Games'}).reset_index()
        
        # Bayesian Formula: (Games * Avg + m * GlobalAvg) / (Games + m)
        stats['Weighted_Difficulty'] = (
            (stats['Games'] * stats['Attacker_Rating']) + (m * global_avg)
        ) / (stats['Games'] + m)
        
        # Save results
        save_path = f"/Users/schoudhry/Desktop/mind-overnight/code/futbolV1/{full_name.replace(' ', '_')}_Full_Report.csv"
        df.to_csv(save_path, index=False)
        
        print("\n--- Final Aggregated Rankings (Hardest Defenders) ---")
        print(f"Global Attacker Average: {global_avg:.2f} | Bayesian Confidence (m): {m}")
        # Sort by lower Weighted Difficulty (harder to score/perform against)
        print(stats.sort_values(by='Weighted_Difficulty').head(10).to_string(index=False))

# Run the research tool
await run_analysis()


--- Researching Rayan Cherki (ID: 979128) ---

Match: Manchester City vs West Ham United
  > Rayan Cherki Rating: 8
  > Rating vs Back Line: 6.23
    - vs Kyle Walker-Peters (West Ham United): 8 | 5.8
    - vs Max Kilman (West Ham United): 8 | 6.3
    - vs Jean-Clair Todibo (West Ham United): 8 | 6
    - vs Oliver Scarles (West Ham United): 8 | 6.3
    - vs Ezra Mayers (West Ham United): 8 | 6.4
    - vs Konstantinos Mavropanos (West Ham United): 8 | 6.6
------------------------------
Match: Nottingham Forest vs Manchester City
  > Rayan Cherki Rating: 8.4
  > Rating vs Back Line: 6.70
    - vs Nicolò Savona (Nottingham Forest): 8.4 | 6.4
    - vs Nikola Milenković (Nottingham Forest): 8.4 | 6.9
    - vs Murillo (Nottingham Forest): 8.4 | 7.3
    - vs Neco Williams (Nottingham Forest): 8.4 | 6.2
------------------------------
Match: Sunderland vs Manchester City
  > Rayan Cherki Rating: 7.6
  > Rating vs Back Line: 7.30
    - vs Trai Hume (Sunderland): 7.6 | 6.6
    - vs Nordi Mukiele

In [1]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import json
from datetime import datetime
from sofascore_wrapper.api import SofascoreAPI
from sofascore_wrapper.search import Search
import nest_asyncio

nest_asyncio.apply()

async def get_sofa_json(page, url):
    await page.goto(url, wait_until="networkidle")
    content = await page.inner_text("body")
    return json.loads(content)

async def run_targeted_analysis():
    # 1. Dynamic Player Search
    target_query = input("Enter the player to analyze: ")
    
    api_wrapper = SofascoreAPI()
    search_engine = Search(api_wrapper, search_string=target_query)
    search_results = await search_engine.search_all()
    
    player_id, full_name = None, None
    for item in search_results.get('results', []):
        if item.get('type') == 'player':
            player_id = item['entity']['id']
            full_name = item['entity']['name']
            break
    
    await api_wrapper.close()
    
    if not player_id:
        print("Player not found.")
        return

    # SETTING THE DATE LIMITS
    start_date_limit = datetime(2024, 1, 1)
    end_date_limit = datetime(2026, 4, 23)
    
    print(f"\n--- Researching {full_name} (Range: Jan 2024 - April 2026) ---")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent="Mozilla/5.0")
        
        all_matches_data = []
        last_match_id = 0
        reached_historical_limit = False
        
        while not reached_historical_limit:
            events_url = f"https://api.sofascore.com/api/v1/player/{player_id}/events/last/{last_match_id}"
            print(f"Fetching batch before ID: {last_match_id if last_match_id != 0 else 'Present'}...")
            
            try:
                events_data = await get_sofa_json(page, events_url)
                events = events_data.get('events', [])
            except: break
            
            if not events: break

            for match in events:
                match_ts = match.get('startTimestamp')
                if not match_ts: continue
                
                match_date = datetime.fromtimestamp(match_ts)

                # SKIP if match is newer than our end limit
                if match_date > end_date_limit:
                    continue
                
                # STOP if match is older than our start limit
                if match_date < start_date_limit:
                    print(f"Reached historical limit ({match_date.date()}). Stopping scrape.")
                    reached_historical_limit = True
                    break
                
                match_id = match['id']
                lineup_url = f"https://api.sofascore.com/api/v1/event/{match_id}/lineups"
                
                try:
                    lineup_data = await get_sofa_json(page, lineup_url)
                    attacker_rating = None
                    target_side = None
                    
                    # Locate Attacker
                    for side in ['home', 'away']:
                        for p_node in lineup_data.get(side, {}).get('players', []):
                            if p_node.get('player', {}).get('id') == player_id:
                                attacker_rating = p_node.get('statistics', {}).get('rating')
                                target_side = side
                                break
                        if target_side: break
                    
                    if attacker_rating:
                        opp_side = 'away' if target_side == 'home' else 'home'
                        opp_team = match[f'{opp_side}Team']['name']
                        
                        active_defenders = []
                        for opp in lineup_data.get(opp_side, {}).get('players', []):
                            d_rating = opp.get('statistics', {}).get('rating')
                            if opp.get('position') == 'D' and d_rating:
                                active_defenders.append({'name': opp['player']['name'], 'rating': d_rating})
                        
                        if active_defenders:
                            backline_avg = sum(d['rating'] for d in active_defenders) / len(active_defenders)
                            print(f"Logged: {match_date.date()} | vs {opp_team} | Rating: {attacker_rating} | Backline Avg: {backline_avg:.2f}")
                            
                            for defender in active_defenders:
                                all_matches_data.append({
                                    'Date': match_date.date(),
                                    'Match': f"{match['homeTeam']['name']} vs {match['awayTeam']['name']}",
                                    'Attacker_Rating': attacker_rating,
                                    'Defender': defender['name'],
                                    'Defender_Team': opp_team,
                                    'Defender_Rating': defender['rating'],
                                    'Backline_Avg': backline_avg
                                })
                except: continue
            
            # Pagination logic
            last_match_id = events[-1]['id']
            if reached_historical_limit: break
            
        await browser.close()

    if all_matches_data:
        df = pd.DataFrame(all_matches_data)
        
        # Bayesian Math
        global_avg = df['Attacker_Rating'].mean()
        m = 3 # Increased confidence factor for longer date range
        
        stats = df.groupby(['Defender', 'Defender_Team']).agg({
            'Attacker_Rating': 'mean',
            'Defender_Rating': 'mean',
            'Defender': 'count'
        }).rename(columns={'Defender': 'Games'}).reset_index()
        
        stats['Weighted_Difficulty'] = ((stats['Games'] * stats['Attacker_Rating']) + (m * global_avg)) / (stats['Games'] + m)
        
        # Save exact filename
        csv_name = f"/Users/schoudhry/Desktop/mind-overnight/code/futbolV1/{full_name.replace(' ', '_')}_vs_Active_Jan24_Apr26.csv"
        df.to_csv(csv_name, index=False)
        
        print(f"\n--- Analysis Complete: {len(df)} 1v1 Data Points ---")
        print(f"File Saved: {csv_name}")
        print("\nTop 10 Hardest Defenders for " + full_name + " (Weighted):")
        print(stats.sort_values(by='Weighted_Difficulty').head(10)[['Defender', 'Defender_Team', 'Games', 'Weighted_Difficulty']])

# Execute
await run_targeted_analysis()


--- Researching Erling Haaland (Range: Jan 2024 - April 2026) ---
Fetching batch before ID: Present...
Logged: 2025-12-06 | vs Sunderland | Rating: 6 | Backline Avg: 6.54
Logged: 2025-12-10 | vs Real Madrid | Rating: 7.1 | Backline Avg: 6.10
Logged: 2025-12-14 | vs Crystal Palace | Rating: 8.4 | Backline Avg: 6.27
Logged: 2025-12-20 | vs West Ham United | Rating: 8.7 | Backline Avg: 6.23
Logged: 2025-12-27 | vs Nottingham Forest | Rating: 6.4 | Backline Avg: 6.70
Logged: 2026-01-01 | vs Sunderland | Rating: 6.2 | Backline Avg: 7.30
Logged: 2026-01-04 | vs Chelsea | Rating: 7 | Backline Avg: 6.56
Logged: 2026-01-07 | vs Brighton & Hove Albion | Rating: 6.8 | Backline Avg: 6.88
Logged: 2026-01-10 | vs Exeter City | Rating: 6.5 | Backline Avg: 4.50
Logged: 2026-01-13 | vs Newcastle United | Rating: 6.5 | Backline Avg: 6.64
Logged: 2026-01-17 | vs Manchester United | Rating: 6.4 | Backline Avg: 7.10
Logged: 2026-01-20 | vs Bodø/Glimt | Rating: 6.2 | Backline Avg: 6.68
Logged: 2026-01-24 |


--- Deep Crawl: Erling Haaland (ID: 839956) ---
Checking Premier League | Season: 25/26...
Checking Premier League | Season: 24/25...
Checking Premier League | Season: 23/24...
Checking World Cup Qual. UEFA | Season: 2026...
Checking EFL Cup | Season: 25/26...
Checking UEFA Champions League | Season: 25/26...
Checking UEFA Champions League | Season: 24/25...
Checking UEFA Champions League | Season: 23/24...
Checking FIFA Club World Cup | Season: 2025...
Checking UEFA Nations League | Season: 24/25...
Checking Community Shield | Season: 24/25...
Checking Community Shield | Season: 23/24...
Checking EURO, Qualification | Season: 2024...
Checking UEFA Super Cup | Season: 23/24...


In [59]:
# Need to work on final bits 
'''
No storing a CSV every time 
Work on the front end of it 
Check how its being store and how to compare the players through scraping 

'''


'\nNo storing a CSV every time \nWork on the front end of it \nCheck how its being store and how to compare the players through scraping \n\n'